# Liu2024 raw inspection notebook

This notebook does **raw inspection only**.

It does **not** preprocess, resample, filter, rereference, window, train, or save a preprocessed dataset.

Purpose:
- inspect what MOABB exposes for Liu2024
- inspect raw sampling frequency, channels, sessions/runs
- inspect annotation names, durations, onset gaps, and trial/event spacing
- compare whether 512 / 536 / 537-sample windows would overlap neighboring annotations
- write a plain text log file you can share back for debugging


In [ ]:

from pathlib import Path
from datetime import datetime
import sys
import math
import json
import platform
import inspect
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

WORKING_DIR = Path.cwd().parent
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
ARTIFACT_DIR = WORKING_DIR / "liu2024_raw_inspection" / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
LOG_PATH = ARTIFACT_DIR / "liu2024_raw_inspection.log"

class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
            stream.flush()

    def flush(self):
        for stream in self.streams:
            stream.flush()

_log_file = open(LOG_PATH, "w", encoding="utf-8")
sys.stdout = Tee(sys.__stdout__, _log_file)
sys.stderr = Tee(sys.__stderr__, _log_file)

def log(msg=""):
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {msg}")

def section(title):
    print("\n" + "=" * 100)
    log(title)
    print("=" * 100)

section("Notebook started")
log(f"Artifact dir: {ARTIFACT_DIR.resolve()}")
log(f"Log path:     {LOG_PATH.resolve()}")
log(f"Python:       {sys.version.replace(chr(10), ' ')}")
log(f"Platform:     {platform.platform()}")



[2026-05-18 12:54:29] Notebook started
[2026-05-18 12:54:29] Artifact dir: /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/liu2024_raw_inspection_artifacts/20260518_125429
[2026-05-18 12:54:29] Log path:     /Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/src/liu2024_raw_inspection_artifacts/20260518_125429/liu2024_raw_inspection.log
[2026-05-18 12:54:29] Python:       3.11.15 (main, Apr  9 2026, 01:18:52) [Clang 21.0.0 (clang-2100.0.123.102)]
[2026-05-18 12:54:29] Platform:     macOS-26.2-arm64-arm-64bit

[2026-05-18 12:54:29] Imports and package versions
[2026-05-18 12:54:29] mne version:        1.11.0


/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-05-18 12:54:30] moabb version:      1.4.3
[2026-05-18 12:54:32] braindecode version:1.4.0

[2026-05-18 12:54:32] Create Liu2024 dataset object directly from MOABB
[2026-05-18 12:54:32] Dataset class:      <class 'moabb.datasets.liu2024.Liu2024'>
[2026-05-18 12:54:32] Dataset repr:       <moabb.datasets.liu2024.Liu2024 object at 0x122a93450>
[2026-05-18 12:54:32] code: Liu2024
[2026-05-18 12:54:32] paradigm: imagery
[2026-05-18 12:54:32] interval: (2, 6)
[2026-05-18 12:54:32] event_id: {'left_hand': 1, 'right_hand': 2}
[2026-05-18 12:54:32] events: {'left_hand': 1, 'right_hand': 2}
[2026-05-18 12:54:32] unit_factor: 1000000.0
[2026-05-18 12:54:32] doi: 10.1038/s41597-023-02787-8
[2026-05-18 12:54:32] subject_list: n=50 values=[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50]
[2026-05-18 12:54:32] sessions_per_subject: <missing>
[2026-05-18 1

/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/moabb/datasets/liu2024.py:412: RuntimeWarning: The unit for channel(s) STI has changed from NA to V.
  raw.set_channel_types(mapping)
/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/moabb/datasets/liu2024.py:412: RuntimeWarning: The unit for channel(s) STI has changed from NA to V.
  raw.set_channel_types(mapping)
/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/moabb/datasets/liu2024.py:412: RuntimeWarning: The unit for channel(s) STI has changed from NA to V.
  raw.set_channel_types(mapping)
/Users/vadim/Documents/School/Spring 2026/CSCE A698 Individual Research/EEG_JEPA/.venv/lib/python3.11/site-packages/moabb/datasets/liu2024.py:412: RuntimeWarning: The unit for channel(s) STI has changed from NA to V.
  raw.set_channel_types(mapping)


[2026-05-18 12:54:58] Loaded data object from MOABB.
[2026-05-18 12:54:58] Top-level type: <class 'dict'>
[2026-05-18 12:54:58] Top-level keys/sample: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

[2026-05-18 12:54:58] Flatten subject/session/run/raw structure
[2026-05-18 12:54:58] Flattened recordings/runs: n=50
[2026-05-18 12:54:58] Example raw[0]: subject=1, session=0, run=0, type=<class 'mne.io.edf.edf.RawEDF'>, n_times=160000, sfreq=500.0
[2026-05-18 12:54:58] Example raw[1]: subject=2, session=0, run=0, type=<class 'mne.io.edf.edf.RawEDF'>, n_times=160000, sfreq=500.0
[2026-05-18 12:54:58] Example raw[2]: subject=3, session=0, run=0, type=<class 'mne.io.edf.edf.RawEDF'>, n_times=160000, sfreq=500.0
[2026-05-18 12:54:58] Example raw[3]: subject=4, session=0, run=0, type=<class 'mne.io.edf.edf.RawEDF'>, n_times=160000, sfreq=500.0
[2026-05-18 12:54:58] Example raw[4]: subject=5, session=0, run=0, type=<class 'mne.io.edf.edf.RawEDF'>, n_times=160000, sfreq=500.0
[2026-05-18 12:54:58] Example raw

In [2]:

section("Imports and package versions")

try:
    import mne
    log(f"mne version:        {mne.__version__}")
except Exception as exc:
    raise RuntimeError(f"Could not import mne: {exc}")

try:
    import moabb
    log(f"moabb version:      {moabb.__version__}")
except Exception as exc:
    raise RuntimeError(f"Could not import moabb: {exc}")

try:
    import braindecode
    log(f"braindecode version:{braindecode.__version__}")
except Exception as exc:
    log(f"braindecode import failed or unavailable: {exc}")

try:
    moabb.set_log_level("INFO")
except Exception:
    pass


In [3]:

section("Create Liu2024 dataset object directly from MOABB")

from moabb import datasets as moabb_datasets

if not hasattr(moabb_datasets, "Liu2024"):
    available = [name for name in dir(moabb_datasets) if "Liu" in name or "liu" in name]
    raise RuntimeError(f"Could not find moabb.datasets.Liu2024. Liu-like dataset names available: {available}")

DatasetClass = getattr(moabb_datasets, "Liu2024")
dataset = DatasetClass()

log(f"Dataset class:      {DatasetClass}")
log(f"Dataset repr:       {dataset!r}")

# Print common MOABB dataset attributes without assuming all exist.
attrs_to_print = [
    "code",
    "paradigm",
    "interval",
    "event_id",
    "events",
    "unit_factor",
    "doi",
    "subject_list",
    "sessions_per_subject",
]
for attr in attrs_to_print:
    value = getattr(dataset, attr, "<missing>")
    if attr == "subject_list" and value != "<missing>":
        log(f"{attr}: n={len(value)} values={value}")
    else:
        log(f"{attr}: {value}")

# Print constructor signature and a small source-location hint.
try:
    log(f"Constructor signature: {inspect.signature(DatasetClass)}")
except Exception as exc:
    log(f"Could not inspect constructor signature: {exc}")

try:
    log(f"Dataset class module: {DatasetClass.__module__}")
    log(f"Dataset class file:   {inspect.getfile(DatasetClass)}")
except Exception as exc:
    log(f"Could not inspect dataset source file: {exc}")


In [4]:

section("Load raw data using dataset.get_data()")

# This intentionally loads all subjects exposed by MOABB for Liu2024.
# No preprocessing is applied in this notebook.
subjects = list(getattr(dataset, "subject_list", []))
if not subjects:
    raise RuntimeError("dataset.subject_list is empty or missing.")

log(f"Subjects to inspect: n={len(subjects)}")
log(f"Subjects: {subjects}")

data = dataset.get_data(subjects=subjects)

log("Loaded data object from MOABB.")
log(f"Top-level type: {type(data)}")
log(f"Top-level keys/sample: {list(data.keys())[:10] if hasattr(data, 'keys') else '<no keys>'}")


In [5]:

section("Flatten subject/session/run/raw structure")

def flatten_moabb_data(data):
    rows = []
    for subject_id, sessions in data.items():
        if not isinstance(sessions, dict):
            rows.append({
                "subject": subject_id,
                "session": None,
                "run": None,
                "raw": sessions,
            })
            continue

        for session_name, runs in sessions.items():
            if isinstance(runs, dict):
                for run_name, raw in runs.items():
                    rows.append({
                        "subject": subject_id,
                        "session": session_name,
                        "run": run_name,
                        "raw": raw,
                    })
            else:
                rows.append({
                    "subject": subject_id,
                    "session": session_name,
                    "run": None,
                    "raw": runs,
                })
    return rows

raw_rows = flatten_moabb_data(data)
log(f"Flattened recordings/runs: n={len(raw_rows)}")

for i, row in enumerate(raw_rows[:10]):
    raw = row["raw"]
    log(
        f"Example raw[{i}]: subject={row['subject']}, session={row['session']}, run={row['run']}, "
        f"type={type(raw)}, n_times={getattr(raw, 'n_times', '<missing>')}, "
        f"sfreq={raw.info.get('sfreq') if hasattr(raw, 'info') else '<missing>'}"
    )

if not raw_rows:
    raise RuntimeError("No raw recordings found after flattening MOABB data.")


In [6]:

section("Raw recording summary: sampling rate, duration, channels, annotations")

recording_summary = []

for row in raw_rows:
    raw = row["raw"]
    sfreq = float(raw.info["sfreq"])
    duration_s = float(raw.n_times / sfreq)
    ch_types = Counter(raw.get_channel_types())
    annotations = raw.annotations

    rec = {
        "subject": str(row["subject"]),
        "session": str(row["session"]),
        "run": str(row["run"]),
        "sfreq": sfreq,
        "n_times": int(raw.n_times),
        "duration_s": duration_s,
        "n_channels": int(len(raw.ch_names)),
        "ch_type_counts": dict(ch_types),
        "n_annotations": int(len(annotations)),
        "annotation_descriptions": sorted(set(map(str, annotations.description))) if len(annotations) else [],
        "first_time": float(getattr(raw, "first_time", 0.0)),
        "meas_date": str(raw.info.get("meas_date")),
        "line_freq": raw.info.get("line_freq"),
        "highpass": raw.info.get("highpass"),
        "lowpass": raw.info.get("lowpass"),
    }
    recording_summary.append(rec)

recording_summary_df = pd.DataFrame(recording_summary)
recording_summary_df.to_csv(ARTIFACT_DIR / "recording_summary.csv", index=False)

log("Recording summary saved to recording_summary.csv")
log("\n" + recording_summary_df.head(20).to_string(index=False))

log("\nSummary by subject:")
subject_recording_summary = (
    recording_summary_df
    .groupby("subject")
    .agg(
        n_recordings=("run", "count"),
        min_sfreq=("sfreq", "min"),
        max_sfreq=("sfreq", "max"),
        min_channels=("n_channels", "min"),
        max_channels=("n_channels", "max"),
        total_annotations=("n_annotations", "sum"),
        total_duration_s=("duration_s", "sum"),
    )
    .reset_index()
)
subject_recording_summary.to_csv(ARTIFACT_DIR / "subject_recording_summary.csv", index=False)
log("\n" + subject_recording_summary.to_string(index=False))

display(recording_summary_df.head(20))


,subject,session,run,sfreq,n_times,duration_s,n_channels,ch_type_counts,n_annotations,annotation_descriptions,first_time,meas_date,line_freq,highpass,lowpass
0,1,0,0,500.0,160000,320.0,32,"{'eeg': 29, 'eog': 2, 'stim': 1}",39,"[left_hand, right_hand]",0.0,2022-12-01 18:58:12+00:00,None,0.0,250.0
1,2,0,0,500.0,160000,320.0,32,"{'eeg': 29, 'eog': 2, 'stim': 1}",39,"[left_hand, right_hand]",0.0,2022-12-01 18:58:15+00:00,None,0.0,250.0
2,3,0,0,500.0,160000,320.0,32,"{'eeg': 29, 'eog': 2, 'stim': 1}",39,"[left_hand, right_hand]",0.0,2022-12-01 18:58:18+00:00,None,0.0,250.0
3,4,0,0,500.0,160000,320.0,32,"{'eeg': 29, 'eog': 2, 'stim': 1}",39,"[left_hand, right_hand]",0.0,2022-12-01 18:58:21+00:00,None,0.0,250.0
4,5,0,0,500.0,160000,320.0,32,"{'eeg': 29, 'eog': 2, 'stim': 1}",39,"[left_hand, right_hand]",0.0,2022-12-01 18:58:23+00:00,None,0.0,250.0
5,6,0,0,500.0,160000,320.0,32,"{'eeg': 29, 'eog': 2, 'stim': 1}",39,"[left_hand, right_hand]",0.0,2022-12-01 18:58:26+00:00,None,0.0,250.0
6,7,0,0,500.0,160000,320.0,32,"{'eeg': 29, 'eog': 2, 'stim': 1}",39,"[left_hand, right_hand]",0.0,2022-12-01 18:58:29+00:00,None,0.0,250.0
7,8,0,0,500.0,160000,320.0,32,"{'eeg': 29, 'eog': 2, 'stim': 1}",39,"[left_hand, right_hand]",0.0,2022-12-01 18:58:31+00:00,None,0.0,250.0
8,9,0,0,500.0,160000,320.0,32,"{'eeg': 29, 'eog': 2, 'stim': 1}",39,"[left_hand, right_hand]",0.0,2022-12-01 18:58:35+00:00,None,0.0,250.0
9,10,0,0,500.0,160000,320.0,32,"{'eeg': 29, 'eog': 2, 'stim': 1}",39,"[left_hand, right_hand]",0.0,2022-12-01 18:58:38+00:00,None,0.0,250.0


In [7]:

section("Channel inspection")

first_raw = raw_rows[0]["raw"]

log(f"First raw subject/session/run: {raw_rows[0]['subject']} / {raw_rows[0]['session']} / {raw_rows[0]['run']}")
log(f"Number of channels: {len(first_raw.ch_names)}")
log("Channel names:")
print(first_raw.ch_names)

log("\nChannel types:")
print(Counter(first_raw.get_channel_types()))

# Dig/montage information
dig = first_raw.info.get("dig")
log(f"\nDigitization points present: {dig is not None}, n={len(dig) if dig is not None else 0}")

try:
    montage = first_raw.get_montage()
    log(f"Montage from raw.get_montage(): {montage}")
    if montage is not None:
        ch_pos = montage.get_positions().get("ch_pos", {})
        log(f"Montage channel positions: n={len(ch_pos)}")
        print(list(ch_pos.keys())[:30])
except Exception as exc:
    log(f"Could not get montage: {exc}")

# Check whether all recordings use identical channel names/order.
channel_name_tuples = Counter(tuple(row["raw"].ch_names) for row in raw_rows)
log(f"\nUnique channel-name/order layouts: {len(channel_name_tuples)}")
for idx, (ch_tuple, count) in enumerate(channel_name_tuples.most_common(5), start=1):
    log(f"Layout {idx}: count={count}, n_channels={len(ch_tuple)}")
    print(list(ch_tuple))


In [8]:

section("Annotation table: every annotation from every raw recording")

annotation_rows = []

for row in raw_rows:
    raw = row["raw"]
    sfreq = float(raw.info["sfreq"])
    ann = raw.annotations

    for i, (onset, duration, desc) in enumerate(zip(ann.onset, ann.duration, ann.description)):
        annotation_rows.append({
            "subject": str(row["subject"]),
            "session": str(row["session"]),
            "run": str(row["run"]),
            "recording_key": f"{row['subject']}::{row['session']}::{row['run']}",
            "annotation_index": int(i),
            "description": str(desc),
            "onset_s": float(onset),
            "duration_s": float(duration),
            "onset_sample_floor": int(math.floor(float(onset) * sfreq)),
            "onset_sample_round": int(round(float(onset) * sfreq)),
            "duration_samples_floor": int(math.floor(float(duration) * sfreq)),
            "duration_samples_round": int(round(float(duration) * sfreq)),
            "sfreq": sfreq,
        })

annotations_df = pd.DataFrame(annotation_rows)
annotations_df.to_csv(ARTIFACT_DIR / "annotations_all.csv", index=False)

log(f"Total annotations: {len(annotations_df)}")
log("Annotation descriptions and counts:")
print(annotations_df["description"].value_counts(dropna=False).to_string())

log("\nAnnotation duration summary by description:")
duration_summary = (
    annotations_df
    .groupby("description")
    .agg(
        n=("duration_s", "count"),
        min_duration_s=("duration_s", "min"),
        median_duration_s=("duration_s", "median"),
        max_duration_s=("duration_s", "max"),
        min_duration_samples_floor=("duration_samples_floor", "min"),
        median_duration_samples_floor=("duration_samples_floor", "median"),
        max_duration_samples_floor=("duration_samples_floor", "max"),
        min_duration_samples_round=("duration_samples_round", "min"),
        median_duration_samples_round=("duration_samples_round", "median"),
        max_duration_samples_round=("duration_samples_round", "max"),
    )
    .reset_index()
    .sort_values(["n", "description"], ascending=[False, True])
)
duration_summary.to_csv(ARTIFACT_DIR / "annotation_duration_summary.csv", index=False)
print(duration_summary.to_string(index=False))

log("\nFirst 50 annotations sorted by subject/session/run/onset:")
print(
    annotations_df
    .sort_values(["subject", "session", "run", "onset_s", "annotation_index"])
    .head(50)
    .to_string(index=False)
)

display(duration_summary)


,description,n,min_duration_s,median_duration_s,max_duration_s,min_duration_samples_floor,median_duration_samples_floor,max_duration_samples_floor,min_duration_samples_round,median_duration_samples_round,max_duration_samples_round
0,left_hand,1000,4.0,4.0,4.0,2000,2000.0,2000,2000,2000.0,2000
1,right_hand,950,4.0,4.0,4.0,2000,2000.0,2000,2000,2000.0,2000


In [9]:

section("Annotation onset-gap inspection")

if annotations_df.empty:
    raise RuntimeError("No annotations found.")

gap_rows = []

for rec_key, g in annotations_df.groupby("recording_key"):
    g = g.sort_values("onset_s").reset_index(drop=True)

    for i in range(len(g) - 1):
        cur = g.iloc[i]
        nxt = g.iloc[i + 1]
        gap_rows.append({
            "recording_key": rec_key,
            "subject": cur["subject"],
            "session": cur["session"],
            "run": cur["run"],
            "from_description": cur["description"],
            "to_description": nxt["description"],
            "from_onset_s": float(cur["onset_s"]),
            "to_onset_s": float(nxt["onset_s"]),
            "gap_s": float(nxt["onset_s"] - cur["onset_s"]),
            "from_duration_s": float(cur["duration_s"]),
            "from_end_s": float(cur["onset_s"] + cur["duration_s"]),
            "annotation_overlap_s": float((cur["onset_s"] + cur["duration_s"]) - nxt["onset_s"]),
        })

gaps_df = pd.DataFrame(gap_rows)
gaps_df.to_csv(ARTIFACT_DIR / "annotation_onset_gaps_all.csv", index=False)

log(f"Total consecutive annotation gaps: {len(gaps_df)}")

if not gaps_df.empty:
    log("Overall onset-gap summary:")
    print(gaps_df["gap_s"].describe(percentiles=[0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]).to_string())

    log("\nGap summary by transition description:")
    transition_summary = (
        gaps_df
        .groupby(["from_description", "to_description"])
        .agg(
            n=("gap_s", "count"),
            min_gap_s=("gap_s", "min"),
            median_gap_s=("gap_s", "median"),
            max_gap_s=("gap_s", "max"),
            n_annotation_overlaps=("annotation_overlap_s", lambda x: int((x > 0).sum())),
            median_annotation_overlap_s=("annotation_overlap_s", "median"),
            max_annotation_overlap_s=("annotation_overlap_s", "max"),
        )
        .reset_index()
        .sort_values("n", ascending=False)
    )
    transition_summary.to_csv(ARTIFACT_DIR / "annotation_transition_gap_summary.csv", index=False)
    print(transition_summary.to_string(index=False))

    log("\nShortest 50 gaps:")
    print(gaps_df.sort_values("gap_s").head(50).to_string(index=False))

    display(transition_summary)


,from_description,to_description,n,min_gap_s,median_gap_s,max_gap_s,n_annotation_overlaps,median_annotation_overlap_s,max_annotation_overlap_s
0,left_hand,right_hand,950,0.002,2.016,4.004,602,1.984,3.998
1,right_hand,left_hand,950,0.040,2.006,4.004,657,1.994,3.960


In [10]:

section("Motor-label focused timing inspection")

# Auto-detect likely left/right labels from annotations and dataset.event_id.
all_descriptions = sorted(set(annotations_df["description"].astype(str)))
event_id = getattr(dataset, "event_id", {}) or {}

candidate_labels = []
for desc in all_descriptions:
    d = desc.lower()
    if "left" in d or "right" in d or "hand" in d or "feet" in d or "foot" in d:
        candidate_labels.append(desc)

for desc in event_id.keys() if isinstance(event_id, dict) else []:
    d = str(desc).lower()
    if ("left" in d or "right" in d or "hand" in d or "feet" in d or "foot" in d) and desc not in candidate_labels:
        candidate_labels.append(str(desc))

if not candidate_labels:
    candidate_labels = all_descriptions

log(f"Candidate motor labels: {candidate_labels}")

motor_df = annotations_df[annotations_df["description"].isin(candidate_labels)].copy()
motor_df = motor_df.sort_values(["subject", "session", "run", "onset_s"]).reset_index(drop=True)
motor_df.to_csv(ARTIFACT_DIR / "annotations_motor_candidates.csv", index=False)

log(f"Motor-candidate annotations: {len(motor_df)}")
log("Motor-candidate counts:")
print(motor_df["description"].value_counts(dropna=False).to_string())

subject_label_counts = (
    motor_df
    .groupby(["subject", "description"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)
subject_label_counts.to_csv(ARTIFACT_DIR / "subject_motor_label_counts.csv", index=False)
log("\nSubject x motor-label counts:")
print(subject_label_counts.to_string(index=False))

display(subject_label_counts.head(20))


description,subject,left_hand,right_hand
0,1,20,19
1,10,20,19
2,11,20,19
3,12,20,19
4,13,20,19
5,14,20,19
6,15,20,19
7,16,20,19
8,17,20,19
9,18,20,19


In [11]:

section("Would 512 / 536 / 537-sample windows overlap neighboring motor annotations?")

WINDOW_SAMPLE_OPTIONS = [512, 536, 537]

overlap_rows = []

for rec_key, g in motor_df.groupby("recording_key"):
    g = g.sort_values("onset_s").reset_index(drop=True)
    if len(g) < 2:
        continue

    sfreq = float(g["sfreq"].iloc[0])

    for win_samples in WINDOW_SAMPLE_OPTIONS:
        win_s = win_samples / sfreq

        for i in range(len(g) - 1):
            cur = g.iloc[i]
            nxt = g.iloc[i + 1]
            gap_s = float(nxt["onset_s"] - cur["onset_s"])
            overlap_s = float((cur["onset_s"] + win_s) - nxt["onset_s"])
            overlap_samples = int(math.ceil(max(0.0, overlap_s) * sfreq))
            overlap_fraction = max(0.0, overlap_s) / win_s

            overlap_rows.append({
                "recording_key": rec_key,
                "subject": cur["subject"],
                "session": cur["session"],
                "run": cur["run"],
                "window_samples": int(win_samples),
                "window_s": float(win_s),
                "from_description": cur["description"],
                "to_description": nxt["description"],
                "from_onset_s": float(cur["onset_s"]),
                "to_onset_s": float(nxt["onset_s"]),
                "gap_s": gap_s,
                "overlap_s": max(0.0, overlap_s),
                "overlap_samples": overlap_samples,
                "overlap_fraction_of_window": overlap_fraction,
                "is_any_overlap": overlap_s > 0,
                "is_major_overlap_25pct": overlap_fraction >= 0.25,
                "is_major_overlap_50pct": overlap_fraction >= 0.50,
            })

window_overlap_df = pd.DataFrame(overlap_rows)
window_overlap_df.to_csv(ARTIFACT_DIR / "window_overlap_diagnostics.csv", index=False)

if window_overlap_df.empty:
    log("No motor window overlaps to inspect.")
else:
    summary = (
        window_overlap_df
        .groupby("window_samples")
        .agg(
            n_pairs=("is_any_overlap", "count"),
            n_any_overlap=("is_any_overlap", "sum"),
            frac_any_overlap=("is_any_overlap", "mean"),
            n_major_overlap_25pct=("is_major_overlap_25pct", "sum"),
            frac_major_overlap_25pct=("is_major_overlap_25pct", "mean"),
            n_major_overlap_50pct=("is_major_overlap_50pct", "sum"),
            frac_major_overlap_50pct=("is_major_overlap_50pct", "mean"),
            min_gap_s=("gap_s", "min"),
            median_gap_s=("gap_s", "median"),
            max_gap_s=("gap_s", "max"),
            median_overlap_s=("overlap_s", "median"),
            max_overlap_s=("overlap_s", "max"),
            median_overlap_fraction=("overlap_fraction_of_window", "median"),
            max_overlap_fraction=("overlap_fraction_of_window", "max"),
        )
        .reset_index()
    )
    summary.to_csv(ARTIFACT_DIR / "window_overlap_summary_by_window_samples.csv", index=False)

    log("Window overlap summary:")
    print(summary.to_string(index=False))

    for win_samples in WINDOW_SAMPLE_OPTIONS:
        log(f"\nWorst overlaps for window_samples={win_samples}:")
        worst = (
            window_overlap_df[window_overlap_df["window_samples"] == win_samples]
            .sort_values(["overlap_fraction_of_window", "overlap_s"], ascending=False)
            .head(30)
        )
        print(worst.to_string(index=False))

    display(summary)


,window_samples,n_pairs,n_any_overlap,frac_any_overlap,n_major_overlap_25pct,frac_major_overlap_25pct,n_major_overlap_50pct,frac_major_overlap_50pct,min_gap_s,median_gap_s,max_gap_s,median_overlap_s,max_overlap_s,median_overlap_fraction,max_overlap_fraction
0,512,1900,3,0.001579,3,0.001579,3,0.001579,0.002,2.006,4.004,0.0,1.022,0.0,0.998047
1,536,1900,4,0.002105,3,0.001579,3,0.001579,0.002,2.006,4.004,0.0,1.070,0.0,0.998134
2,537,1900,4,0.002105,3,0.001579,3,0.001579,0.002,2.006,4.004,0.0,1.072,0.0,0.998138


In [12]:

section("Equivalent MNE events_from_annotations inspection")

# This helps inspect whether MNE event extraction maps annotations as expected.
mne_event_rows = []

for row in raw_rows[:10]:
    raw = row["raw"]
    try:
        events, event_id_map = mne.events_from_annotations(raw, verbose=False)
        log(
            f"subject={row['subject']} session={row['session']} run={row['run']} "
            f"events.shape={events.shape}, event_id_map={event_id_map}"
        )

        for i, ev in enumerate(events[:20]):
            mne_event_rows.append({
                "subject": str(row["subject"]),
                "session": str(row["session"]),
                "run": str(row["run"]),
                "event_index": int(i),
                "sample": int(ev[0]),
                "previous_value": int(ev[1]),
                "event_code": int(ev[2]),
            })
    except Exception as exc:
        log(f"events_from_annotations failed for subject={row['subject']} session={row['session']} run={row['run']}: {exc}")

mne_events_preview_df = pd.DataFrame(mne_event_rows)
mne_events_preview_df.to_csv(ARTIFACT_DIR / "mne_events_preview_first_10_recordings.csv", index=False)

if not mne_events_preview_df.empty:
    log("\nMNE events preview:")
    print(mne_events_preview_df.head(100).to_string(index=False))


In [13]:

section("Final artifact list")

artifacts = sorted(ARTIFACT_DIR.glob("*"))
for path in artifacts:
    log(f"{path.name}  ({path.stat().st_size} bytes)")

log("\nDONE.")
log(f"Send back this log file if you want me to inspect the output: {LOG_PATH.resolve()}")

# Restore stdout/stderr handles at the very end if desired.
# Leave log file open until notebook kernel exits so all output is captured.
